# Futures Desk: Bond, IR, Equity & Commodity

Complete futures workflow:
1. Build a curve from SOFR futures (with convexity adjustment)
2. Bond futures basis trade (CTD, implied repo, hedge ratio)
3. IR futures strip (pack/bundle/butterfly, Fed implied path)
4. Commodity term structure (contango, roll yield, calendar spread)
5. Multi-asset book (margin, risk, stress, roll schedule)

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), "python"))

from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
import numpy as np

from pricebook.bond import FixedRateBond
from pricebook.bootstrap import bootstrap
from pricebook.discount_curve import DiscountCurve
from pricebook.ir_futures import (
    IRFuture, FuturesType, hw_convexity_adjustment,
    futures_strip_rates, futures_pack, futures_bundle, futures_butterfly,
    fed_funds_implied_probability,
)
from pricebook.futures_bootstrap import futures_strip
from pricebook.bond_futures import (
    DeliverableBond, conversion_factor, cheapest_to_deliver,
    implied_repo_rate, bond_futures_basis, delivery_basket,
    futures_hedge_ratio, tail_adjusted_hedge_ratio, forward_bond_price,
)
from pricebook.bond_futures_options import (
    end_of_month_option, quality_option, timing_option,
    net_basis_decomposition, joint_delivery_option_value,
)
from pricebook.futures import (
    EquityFuture, CommodityFuture,
    contango_or_backwardation, roll_yield, calendar_spread,
    futures_strip_curve, implied_convenience_yield,
)
from pricebook.futures_desk import (
    FuturesBook, FuturesBookEntry, FuturesAssetClass, BondFuture,
    futures_risk_metrics, futures_margin_check, futures_carry_roll,
    futures_dashboard, futures_stress_suite, futures_capital,
    futures_hedge_recommendations, FuturesLifecycle,
    roll_schedule, futures_cash_basis_rv,
)
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 7, 15)
print(f"Valuation date: {REF}")

## 1. Build curve from SOFR futures

Three-phase bootstrap: deposits (short end) → SOFR futures with HW convexity (middle) → swaps (long end).

In [ ]:
# Market data: Jul 2024 levels
deposits = [
    (REF + relativedelta(months=1), 0.0533),
    (REF + relativedelta(months=3), 0.0531),
]

# SOFR 3M futures (IMM dates, quoted as 100 - rate)
# Convert to (accrual_start, accrual_end, rate)
futures_quotes = [
    (date(2024, 9, 18), date(2024, 12, 18), 0.0510),
    (date(2024, 12, 18), date(2025, 3, 19), 0.0480),
    (date(2025, 3, 19), date(2025, 6, 18), 0.0450),
    (date(2025, 6, 18), date(2025, 9, 17), 0.0420),
    (date(2025, 9, 17), date(2025, 12, 17), 0.0400),
    (date(2025, 12, 17), date(2026, 3, 18), 0.0385),
    (date(2026, 3, 18), date(2026, 6, 17), 0.0375),
    (date(2026, 6, 17), date(2026, 9, 16), 0.0370),
]

swaps = [
    (REF + relativedelta(years=5), 0.0415),
    (REF + relativedelta(years=7), 0.0405),
    (REF + relativedelta(years=10), 0.0395),
]

# Build curve with HW convexity adjustment
curve = futures_strip(
    REF, deposits, futures_quotes, swaps,
    hw_a=0.03, hw_sigma=0.01,
)

# Show the futures strip
ir_futures = [IRFuture(s, e, FuturesType.SOFR_3M) for s, e, _ in futures_quotes]
strip = futures_strip_rates(ir_futures, curve, a=0.03, sigma=0.01)

print(f"{'Period':>25}  {'Forward':>8}  {'CA (bp)':>8}  {'Fut Rate':>8}  {'Price':>8}")
print(f"{'─'*25}  {'─'*8}  {'─'*8}  {'─'*8}  {'─'*8}")
for s in strip:
    print(f"{str(s['start'])} → {str(s['end'])[:5]}  {s['forward']*100:>7.2f}%  "
          f"{s['convexity']*1e4:>7.2f}  {s['futures_rate']*100:>7.2f}%  {s['price']:>8.3f}")

# Compare: curve with vs without convexity
curve_no_ca = futures_strip(REF, deposits, futures_quotes, swaps)
print(f"\nCurve zero rates at 2Y:")
d2y = REF + relativedelta(years=2)
print(f"  With HW convexity:    {curve.zero_rate(d2y)*100:.3f}%")
print(f"  Without convexity:    {curve_no_ca.zero_rate(d2y)*100:.3f}%")
print(f"  Difference:           {(curve.zero_rate(d2y) - curve_no_ca.zero_rate(d2y))*1e4:.2f}bp")

## 2. Bond futures basis trade

US Treasury futures (ZN — 10Y note future): delivery basket analysis, CTD, implied repo, hedge ratio.

In [ ]:
# ZN delivery basket: deliverable 10Y T-Notes
# Conversion factors at 6% standard yield
deliverables = [
    DeliverableBond(
        bond=FixedRateBond.treasury_note(date(2021, 2, 15), date(2031, 2, 15), 0.0125),
        market_price=86.50,
        conversion_factor=conversion_factor(0.0125, 6.5),
        accrued=0.52,
    ),
    DeliverableBond(
        bond=FixedRateBond.treasury_note(date(2021, 8, 15), date(2031, 8, 15), 0.0125),
        market_price=86.00,
        conversion_factor=conversion_factor(0.0125, 7.0),
        accrued=0.52,
    ),
    DeliverableBond(
        bond=FixedRateBond.treasury_note(date(2024, 2, 15), date(2034, 2, 15), 0.0425),
        market_price=102.50,
        conversion_factor=conversion_factor(0.0425, 9.5),
        accrued=1.77,
    ),
]

futures_price = 110.50  # ZN Sep 2024

# CTD identification (quick — by gross basis)
ctd = cheapest_to_deliver(deliverables, futures_price)
print(f"CTD (gross basis): bond #{ctd.ctd_index} — gross basis = {ctd.gross_basis:.3f}")

# Full basket analysis (by implied repo — more accurate)
basket = delivery_basket(
    deliverables, futures_price,
    accrued_at_delivery=[d.accrued + 0.3 for d in deliverables],  # delivery accrued
    coupon_incomes=[d.bond.coupon_rate * 100 * 60/365 for d in deliverables],  # ~2 months carry
    days_to_delivery=60,
    accrued_at_purchase=[d.accrued for d in deliverables],
)

print(f"\n{'Bond':>6}  {'Cpn':>5}  {'Price':>7}  {'CF':>6}  {'Gross':>7}  {'Impl Repo':>10}")
print(f"{'─'*6}  {'─'*5}  {'─'*7}  {'─'*6}  {'─'*7}  {'─'*10}")
for b in basket.bonds:
    d = deliverables[b.index]
    print(f"{'→' if b.index == basket.ctd_index else ' '} #{b.index}  "
          f"{d.bond.coupon_rate*100:>4.2f}%  {d.market_price:>7.2f}  {d.conversion_factor:>6.4f}  "
          f"{b.gross_basis:>+6.3f}  {b.implied_repo*100:>9.2f}%")

# Hedge ratio
ctd_bond = deliverables[basket.ctd_index]
hedge_bond_dv01 = 0.085  # DV01 of the bond to hedge (per 100 face)
ctd_dv01 = 0.065         # DV01 of the CTD
hr = futures_hedge_ratio(hedge_bond_dv01, ctd_dv01, ctd_bond.conversion_factor)
hr_tail = tail_adjusted_hedge_ratio(hedge_bond_dv01, ctd_dv01, ctd_bond.conversion_factor,
                                     repo_rate=0.053, days_to_delivery=60)
print(f"\nHedge ratio: {hr:.4f} (tail-adjusted: {hr_tail:.4f})")
print(f"To hedge $10M face: {hr * 100:.0f} contracts (tail: {hr_tail * 100:.0f})")

## 3. Delivery option decomposition

In [ ]:
# Delivery option components
eom = end_of_month_option(
    futures_dv01=0.065 * 1000,  # DV01 in $ per contract
    daily_vol_bps=5.0,
    n_business_days=5,
    coupon_accrual_per_day=0.0425 * 100 / 365,
)
print(f"EOM (wild card) option: {eom.value:.2f} per contract")

# Quality option needs dict of {bond_name: gross_basis}
basis_dict = {f"bond_{b.index}": b.gross_basis for b in basket.bonds}
qual = quality_option(
    deliverable_bases=basis_dict,
    yield_vol_bps=5.0,
    futures_dv01=0.065 * 1000,
)
print(f"Quality option: {qual.value:.2f} (switch prob: {qual.ctd_switch_probability*100:.1f}%)")

timing = timing_option(
    futures_price=futures_price,
    coupon_rate=ctd_bond.bond.coupon_rate,
    repo_rate=0.053,
    conversion_factor=ctd_bond.conversion_factor,
)
print(f"Timing option: {timing.value:.2f} (optimal delivery: day {timing.optimal_delivery_day})")

joint = joint_delivery_option_value(eom, qual, timing)
print(f"\nJoint delivery option: {joint.total_value:.2f}")
print(f"  EOM:     {joint.eom_value:.2f}")
print(f"  Quality: {joint.quality_value:.2f}")
print(f"  Timing:  {joint.timing_value:.2f}")

## 4. IR futures strip — pack, bundle, butterfly, Fed path

In [ ]:
# White pack (first 4 quarters)
pack = futures_pack(ir_futures[:4], curve, a=0.03, sigma=0.01)
print(f"White pack: price={pack.price:.3f}  rate={pack.implied_rate*100:.2f}%  DV01=${pack.dv01:.0f}")

# 2Y bundle (all 8 quarters)
bundle = futures_bundle(ir_futures, curve, a=0.03, sigma=0.01)
print(f"2Y bundle: price={bundle.price:.3f}  rate={bundle.implied_rate*100:.2f}%  {bundle.years}Y")

# Butterfly on quarters 1-2-3
bf = futures_butterfly(ir_futures[0], ir_futures[1], ir_futures[2], curve)
print(f"Butterfly (Q1-Q2-Q3): spread={bf.spread:.3f}")
print(f"  Front={bf.front_price:.3f}  Mid={bf.mid_price:.3f}  Back={bf.back_price:.3f}")

# Fed funds implied path
print(f"\nFed funds implied path (from SOFR futures):")
current_ffr = 0.0533
for s in strip:
    prob = fed_funds_implied_probability(s['futures_rate'], current_ffr, move_size=-0.0025)
    cuts = (current_ffr - s['futures_rate']) / 0.0025
    print(f"  {s['start']} → {s['end']}: rate={s['futures_rate']*100:.2f}%  "
          f"implied cuts={cuts:.1f}  prob(next cut)={min(prob, 1)*100:.0f}%")

# Plot: futures strip + implied rate path
with apply_theme():
    fig, (ax1, ax2) = create_figure(2)
    
    prices = [s['price'] for s in strip]
    rates = [s['futures_rate'] * 100 for s in strip]
    labels = [str(s['start'])[5:] for s in strip]
    
    ax1.bar(labels, prices, alpha=0.8)
    ax1.set_ylabel('Futures Price')
    ax1.set_title('SOFR Futures Strip')
    ax1.tick_params(axis='x', rotation=45)
    
    ax2.plot(labels, rates, 'o-', linewidth=2)
    ax2.axhline(current_ffr * 100, color='red', linestyle='--', label=f'Current FFR ({current_ffr*100:.2f}%)')
    ax2.set_ylabel('Implied Rate (%)')
    ax2.set_title('Implied Rate Path')
    ax2.legend(fontsize=8)
    ax2.tick_params(axis='x', rotation=45)
    
    fig.tight_layout()

## 5. Commodity term structure

In [ ]:
# WTI crude oil futures curve (Jul 2024 levels)
wti_spot = 80.50
wti_futures = [
    CommodityFuture("WTI", date(2024, 8, 20), 80.20),
    CommodityFuture("WTI", date(2024, 9, 20), 79.50),
    CommodityFuture("WTI", date(2024, 10, 22), 78.80),
    CommodityFuture("WTI", date(2024, 11, 20), 78.00),
    CommodityFuture("WTI", date(2024, 12, 19), 77.20),
    CommodityFuture("WTI", date(2025, 3, 20), 75.50),
    CommodityFuture("WTI", date(2025, 6, 20), 74.00),
    CommodityFuture("WTI", date(2025, 12, 19), 72.00),
]

# Term structure
structure = contango_or_backwardation(wti_futures[0].price, wti_futures[-1].price)
print(f"WTI term structure: {structure}")

print(f"\n{'Expiry':>12}  {'Price':>7}  {'Roll Yield':>10}  {'Conv Yield':>10}")
print(f"{'─'*12}  {'─'*7}  {'─'*10}  {'─'*10}")
for i, f in enumerate(wti_futures):
    ry = roll_yield(wti_futures[0].price, f.price) if i > 0 else 0.0
    T = (f.expiry - REF).days / 365.25
    cy = implied_convenience_yield(wti_spot, f.price, 0.053, 0.02, T) if T > 0 else 0.0
    print(f"{f.expiry.isoformat():>12}  {f.price:>7.2f}  {ry*100:>+9.1f}%  {cy*100:>+9.2f}%")

# Calendar spread
cs = calendar_spread(wti_futures[0], wti_futures[4])
print(f"\nCalendar spread (Aug-Dec): {cs.spread:+.2f} ({cs.structure})")
print(f"Roll yield: {cs.roll_yield*100:+.1f}% annualised")

# Plot
with apply_theme():
    fig, ax = create_figure(1)
    expiries = [f.expiry for f in wti_futures]
    prices = [f.price for f in wti_futures]
    ax.plot(expiries, prices, 'o-', linewidth=2, label='WTI Futures')
    ax.axhline(wti_spot, color='red', linestyle='--', label=f'Spot ({wti_spot})')
    ax.set_xlabel('Expiry')
    ax.set_ylabel('Price ($/bbl)')
    ax.set_title('WTI Crude Oil Futures Curve (Backwardation)')
    ax.legend(fontsize=9)
    fig.tight_layout()

## 6. Multi-asset futures book — margin, risk, stress, roll

In [ ]:
# Build a multi-asset book
book = FuturesBook("trading_book")

# Bond future: long 50 ZN contracts
zn = BondFuture(trade_price=110.00, market_price=futures_price, expiry=date(2024, 9, 19),
                ctd_dv01=0.065, ctd_cf=ctd_bond.conversion_factor)
book.add(FuturesBookEntry("zn_sep24", zn, 50, FuturesAssetClass.BOND,
                          exchange="CBOT", margin_per_contract=2_000))

# IR future: short 100 SOFR 3M (positioning for rate cuts)
sofr = ir_futures[2]  # Mar 2025
book.add(FuturesBookEntry("sofr_mar25", sofr, -100, FuturesAssetClass.IR,
                          exchange="CME", margin_per_contract=1_000))

# Equity future: long 10 ES (S&P 500)
es = EquityFuture(spot=5500, expiry=date(2024, 9, 20), rate=0.053,
                  div_yield=0.014, notional_per_point=50)
book.add(FuturesBookEntry("es_sep24", es, 10, FuturesAssetClass.EQUITY,
                          exchange="CME", margin_per_contract=15_000))

# Summary
print(f"Book: {len(book)} positions, {book.total_contracts()} contracts")
print(f"Total margin: ${book.total_margin():,.0f}")

# By asset class
for ac, entries in book.by_asset_class().items():
    contracts = sum(e.contracts for e in entries)
    margin = sum(e.margin_per_contract * abs(e.contracts) for e in entries)
    print(f"  {ac.value:>10}: {contracts:>+5} contracts, ${margin:>10,.0f} margin")

In [ ]:
# Dashboard
dash = futures_dashboard(book, REF)
print(f"Futures Dashboard — {dash.date}")
print(f"  Positions:    {dash.n_positions}")
print(f"  Total PV:     ${dash.total_pv:>+12,.0f}")
print(f"  Total DV01:   ${dash.total_dv01:>+12,.0f}")
print(f"  Total Margin: ${dash.total_margin:>12,.0f}")

# Stress test
print(f"\nStress Scenarios:")
print(f"{'Scenario':>20}  {'Description':>30}  {'P&L':>12}")
print(f"{'─'*20}  {'─'*30}  {'─'*12}")
for s in futures_stress_suite(book):
    print(f"{s.scenario:>20}  {s.description:>30}  ${s.pnl:>+11,.0f}")

# Margin check
variation = 25_000  # today's settlement P&L
margin = futures_margin_check(variation, book.total_margin(), book.total_margin() * 0.7)
print(f"\nMargin check: total=${margin.total_margin:,.0f} call=${margin.margin_call:,.0f}")

In [ ]:
# Roll schedule
rolls = roll_schedule(book, REF, roll_days_before=70)
print(f"Roll recommendations ({len(rolls)}):")
for r in rolls:
    print(f"  {r.trade_id}: {r.action} ({r.days_to_expiry}d to expiry)")

# Cross-market RV: SOFR futures vs swap rate
sofr_fut_rate = strip[2]['futures_rate']  # Mar 2025 SOFR future
swap_2y_rate = 0.0460  # 2Y swap rate
rv = futures_cash_basis_rv(sofr_fut_rate, swap_2y_rate,
                           historical_mean=0.0002, historical_std=0.0003)
print(f"\nFutures-Cash RV:")
print(f"  SOFR future rate: {sofr_fut_rate*100:.2f}%")
print(f"  2Y swap rate:     {swap_2y_rate*100:.2f}%")
print(f"  Basis:            {rv.spread*1e4:+.1f}bp")
print(f"  Z-score:          {rv.z_score:+.1f}")
print(f"  Signal:           {rv.signal}")

## Summary

| Module | What it does |
|--------|-------------|
| `futures_bootstrap` | Build discount curve from deposits + futures + swaps with HW convexity |
| `ir_futures` | SOFR/Euribor futures, strips, packs, bundles, butterflies, Fed implied |
| `bond_futures` | CTD, conversion factors, implied repo, basis, hedge ratios |
| `bond_futures_options` | Delivery options: EOM, quality, timing, net basis decomposition |
| `futures` | Equity + commodity futures, contango, roll yield, convenience yield |
| `futures_desk` | Unified book, margin, risk, stress, capital, roll schedule, cross-market RV |

**Key concepts:**
- **Convexity adjustment**: futures rate > forward rate (daily margining bias) — use HW analytical formula
- **CTD identification**: by highest implied repo rate (not gross basis) — use `delivery_basket()`
- **Net basis ≈ delivery option value**: gross basis - carry = embedded optionality
- **Pack/bundle**: standardised multi-quarter structures for curve exposure
- **Backwardation**: convenience yield > storage + financing — positive roll yield
- **Futures-cash basis**: spread between futures-implied and swap rates — RV signal